# Train model introduction

Train a small PyMC MFM network from scratch on ten synthetic marketing examples. Unlike the [load model introduction](river_1_quickstart.ipynb), this notebook starts with untrained weights. Five updates are enough to see how training works, not enough to use the model for forecasts.

In a PyMC MFM repository checkout with Python 3.13 or newer, run `uv sync --locked --group generation` to install PyMC MFM and PyMC Generator. Use the checkout's `.venv` as your notebook kernel. No model downloads, credentials, or GPU are needed.

## 1. Generate ten examples

Each example is a simulated world with 16 weeks of marketing data, two media channels, two controls, and one hidden factor. Generate one world for each of ten independently sampled setups (`n_cells=10`, `draws_per_cell=1`).

In [1]:
from pymc_generator import DataGenerator, make_scm_prior

prior = make_scm_prior(
    n_cells=10,
    draws_per_cell=1,
    n_time_steps=16,
    n_treatments=2,
    n_covariates=2,
    n_latent=1,
    nonlinearity="linear",
    seed=0,
)
corpus = DataGenerator(prior).generate()

The generator marks which worlds to hold out. The trainer updates its weights on the other worlds and uses held-out worlds for evaluation.

In [2]:
held_out = corpus["is_val"].astype(bool)
print(f"{(~held_out).sum()} training worlds, {held_out.sum()} held-out worlds")

8 training worlds, 2 held-out worlds


## 2. Size the network

Use one attention layer, one head, and a hidden width of eight. The number of controls and hidden factors must match the generated corpus.

In [3]:
from pymc_mfm.model import ModelConfig

model_config = ModelConfig(
    d_model=8,
    n_layers=1,
    n_heads=1,
    d_ff=16,
    flow_hidden=16,
    n_controls=prior.n_covariates,
    n_demand=prior.n_latent,
)

## 3. Set the training budget

Run five updates with two worlds per batch and a constant learning rate. Record the loss every step and evaluate only after the last one.

In [4]:
from pymc_mfm.training import TrainingConfig

training = TrainingConfig(
    steps=5,
    batch_size=2,
    lr_schedule="constant",
    warmup_steps=0,
    log_every=1,
    eval_every=5,
    eval_worlds=1,
    eval_samples=4,
    milestone_steps=(),
)

## 4. Train on CPU

Pass the in-memory corpus to `train`. Disable MLflow tracking for this example: the trainer still saves the run and model locally. A fresh temporary directory lets you rerun the cell without overwriting an earlier run.

In [5]:
from tempfile import mkdtemp

from pymc_mfm.training import TrackingConfig, train

result = train(
    [corpus],
    out_dir=mkdtemp(prefix="mfm-train-"),
    model_config=model_config,
    training=training,
    tracking=TrackingConfig(enabled=False),
    device="cpu",
)

## 5. Inspect the run

Read the five recorded training losses. Each step can use a different batch, so the loss need not go down at every step.

In [6]:
for record in result.records().training["records"]:
    print(f"Step {record['step']}: loss = {record['loss']:.3f}")

Step 1: loss = 4.138
Step 2: loss = 4.945
Step 3: loss = 5.111
Step 4: loss = 6.282
Step 5: loss = 5.057


The trainer exports the final weights as a local bundle in `result.run_dir / "model"`. The [training guide](../docs/guide/training.qmd) describes the rest of the run files and the settings for longer runs.

In [7]:
model_dir = result.run_dir / "model"
print("Bundle files:", ", ".join(sorted(file.name for file in model_dir.iterdir())))

Bundle files: config.json, manifest.json, model.safetensors
